# Análisis ConnectaTel

El **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamérica, ConnectaTel.

Trabajaremos con información registrada **hasta el año 2024**, lo cual permitirá analizar el comportamiento del negocio dentro de ese periodo.

Para ello se trabajará con tres datasets:  

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)  
- **users.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)  
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)  

---
## 🧩 Paso 1: Cargar y explorar

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

In [ ]:
# importar librerías
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# cargar archivos
plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

In [ ]:
plans.head(5)

,plan_name,messages_included,gb_per_month,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute
0,Basico,100,5,100,12,1.2,0.08,0.10
1,Premium,500,20,600,25,1.0,0.05,0.07


In [ ]:
users.head(5)

,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
0,10000,Carlos,Garcia,38,Medellín,2022-01-01 00:00:00.000000000,Basico,NaN
1,10001,Mateo,Torres,53,?,2022-01-01 06:34:17.914478619,Basico,NaN
2,10002,Sofia,Ramirez,57,CDMX,2022-01-01 13:08:35.828957239,Basico,NaN
3,10003,Mateo,Ramirez,69,Bogotá,2022-01-01 19:42:53.743435858,Premium,NaN
4,10004,Mateo,Torres,63,GDL,2022-01-02 02:17:11.657914478,Basico,NaN


In [ ]:
usage.head(5)

,id,user_id,type,date,duration,length
0,1,10332,call,2024-01-01 00:00:00.000000000,0.09,NaN
1,2,11458,text,2024-01-01 00:06:30.969774244,NaN,39.0
2,3,11777,text,2024-01-01 00:13:01.939548488,NaN,36.0
3,4,10682,call,2024-01-01 00:19:32.909322733,1.53,NaN
4,5,12742,call,2024-01-01 00:26:03.879096977,4.84,NaN


### 1.2 Exploración de la estructura de los datasets

**🎯 Objetivo:**  
Conocer la **estructura de cada dataset**, revisar cuántas filas y columnas tienen, identificar los **tipos de datos** de cada columna y detectar posibles **inconsistencias o valores nulos** antes de iniciar el análisis.  

In [ ]:
# revisar el número de filas y columnas de cada dataset
print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

plans (2, 8)
users (4000, 8)
usage (40000, 6)


In [ ]:
plans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   plan_name          2 non-null      object 
 1   messages_included  2 non-null      int64  
 2   gb_per_month       2 non-null      int64  
 3   minutes_included   2 non-null      int64  
 4   usd_monthly_pay    2 non-null      int64  
 5   usd_per_gb         2 non-null      float64
 6   usd_per_message    2 non-null      float64
 7   usd_per_minute     2 non-null      float64
dtypes: float64(3), int64(4), object(1)
memory usage: 256.0+ bytes


In [ ]:
users.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4000 entries, 0 to 3999
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     4000 non-null   int64 
 1   first_name  4000 non-null   object
 2   last_name   4000 non-null   object
 3   age         4000 non-null   int64 
 4   city        3531 non-null   object
 5   reg_date    4000 non-null   object
 6   plan        4000 non-null   object
 7   churn_date  466 non-null    object
dtypes: int64(2), object(6)
memory usage: 250.1+ KB


In [ ]:
usage.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        40000 non-null  int64  
 1   user_id   40000 non-null  int64  
 2   type      40000 non-null  object 
 3   date      39950 non-null  object 
 4   duration  17924 non-null  float64
 5   length    22104 non-null  float64
dtypes: float64(2), int64(2), object(2)
memory usage: 1.8+ MB


---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

In [ ]:
# cantidad de nulos para users
print(users.isna().sum())
print()
print(users.isna().mean().sort_values(ascending=False))

user_id          0
first_name       0
last_name        0
age              0
city           469
reg_date         0
plan             0
churn_date    3534
dtype: int64

churn_date    0.88350
city          0.11725
user_id       0.00000
first_name    0.00000
last_name     0.00000
age           0.00000
reg_date      0.00000
plan          0.00000
dtype: float64


In [ ]:
# cantidad de nulos para usage
print(usage.isna().sum())
print()
print(usage.isna().mean().sort_values(ascending=False))

id              0
user_id         0
type            0
date           50
duration    22076
length      17896
dtype: int64

duration    0.55190
length      0.44740
date        0.00125
id          0.00000
user_id     0.00000
type        0.00000
dtype: float64


✍️ **Comentario**: En  general, los datasets presenta un nivel medio de valores faltantes. Las variables con mayor porción de missing values son:

users =
- churn_date(88%): La ausencia aqui es la mayor pero no es tan relevante, ya que, los datos faltantes se deben a que los clientes no tienen fecha de cancelación sino al contrario, siguen activos. Por lo tanto, se dejan los valores como nulos.
- city(11%): Los datos de ubicación también presentan valores faltantes moderados. Esto podría impactar en análisis geográficos o reportes por región. Dejar como nulos.

usage=
- duration(55%) y lengh(44%): La ausencia aquí se relaciona a que la duración es en las llamdas y la longitud en los mensajes. Por lo tanto, se ignoran los valores faltantes.
- date(0.12%):La ausencia aquí es mínima, pero relevante, ya que podría afectar series temporales o análisis de uso por fecha. Aquí se debe imputar.

 ---

### 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

In [ ]:
users.describe()

,user_id,age
count,4000.000000,4000.000000
mean,11999.500000,33.739750
std,1154.844867,123.232257
min,10000.000000,-999.000000
25%,10999.750000,32.000000
50%,11999.500000,47.000000
75%,12999.250000,63.000000
max,13999.000000,79.000000


- La columna `user_id` presenta std alta o con mucha variabilidad y un rango corto.
- La columna `age` presenta mucha variabilidad y un amplio rango con posibles outliers.

In [ ]:
usage.describe()

- Las columnas `id` y `user_id`presentan alta variabilidad sin posibles outliers.
- La columnas `duration` y `length` presentan baja variabilidad con rango amplio.

In [ ]:
# explorar columnas categóricas de users
columnas_user = ['city', 'plan']
for columna in columnas_user:
    print(f"=={columna.capitalize()} Frecuencia Absoluta==")
    print(users[columna].value_counts())
    print(f"=={columna.capitalize()} Frecuencia Relativa==")
    print(users[columna].value_counts(normalize=True))
    print()

- La columna `city` tiene 7 categorías siendo `Bogotá` la más frecuente
- La columna `plan` presenta dos categorias siendo `Basico` dominante con el 65% de participación.

In [ ]:
# explorar columna categórica de usage
usage['type']
print("== Type Frecuencia Absoluta ==")
print(usage['type'].value_counts())
print("== Type Frecuencia Relativa ==")
print(usage['type'].value_counts(normalize=True))

== Type Frecuencia Absoluta ==
text    22092
call    17908
Name: type, dtype: int64
== Type Frecuencia Relativa ==
text    0.5523
call    0.4477
Name: type, dtype: float64


- La columna `type` tiene dos categorías siendo `text` la mas frencuente.

---
✍️ **Comentario**:
- La columna `city`presenta 96 valores inválidos se recomienda estandarizar.
- La columna `age` presenta 55 sentinels se recomienda estandarizar.  

### 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

In [ ]:
# Convertir a fecha la columna `reg_date` de users
users['reg_date'] = pd.to_datetime(users['reg_date'],errors='coerce')
print(users['reg_date'])

In [ ]:
# Convertir a fecha la columna `date` de usage
usage['date'] = pd.to_datetime(usage['date'],errors='coerce')
print(usage['date'])

In [ ]:
# Revisar los años presentes en `reg_date` de users
users['reg_date'].dt.year.value_counts().sort_index()

En `reg_date`, presenta distribución equilibrada y crecimiento gradual año tras año. Además presenta 40 fechas imposibles, recomiendo estandarizarlos marcandolos como nulos e investigar si hay un patrón.

In [ ]:
# Revisar los años presentes en `date` de usage
usage['date'].dt.year.value_counts().sort_index()

En `date`, tiene un volumen muy alto de registros para un solo año, 2024 es un año válido y este parece ser el año principal del dataset `usage`.

✍️ **Comentario**:
- Aparecen años sin transcurrir en la columna `reg_date` del dataframe `users`, se recomienda marcarlos como nulos e investigar si hay un patrón.
- La columna `date` del dataframe `usage` presenta valores nulos, se pueden dejar como nulos pero si se requiere analisis por fechas mejor realizar imputación.

---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

In [ ]:
# Reemplazar -999 por la mediana de age
age_mediana = users['age'].median()
users['age'] = users['age'].replace(-999,age_mediana)

# Verificar cambios
users['age'].describe()

In [ ]:
# Reemplazar ? por NA en city
users['city']= users['city'].replace('?',pd.NA)

# Verificar cambios
users['city'].describe()

In [ ]:
# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'].dt.year == 2026, 'reg_date'] = pd.NaT
# Verificar cambios
users['reg_date'].describe()

### 3.2 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Decidir qué hacer con los valores nulos según su proporción y relevancia.

In [ ]:
# Verificación MAR en usage (Missing At Random) para duration
# Crear columna indicadora de valores faltantes
usage['duration_missing'] = usage['duration'].isna()
# Tabla cruzada entre type y valores faltantes de duration
tabla_cruzada = pd.crosstab(usage['type'], usage['duration_missing'])
print(tabla_cruzada)
# Tabla cruzada con porcentajes
tabla_porcentajes = pd.crosstab(usage['type'], usage['duration_missing'], normalize='index') * 100
print("\nPorcentajes por tipo:")
print(tabla_porcentajes)

In [ ]:
# Verificación MAR en usage (Missing At Random) para length
# Crear columna indicadora de valores faltantes
usage['length_missing'] = usage['length'].isna()
# Tabla cruzada entre type y valores faltantes de length
tabla_cruzada = pd.crosstab(usage['type'], usage['length_missing'])
print(tabla_cruzada)
# Tabla cruzada con porcentajes
tabla_porcentajes = pd.crosstab(usage['type'], usage['length_missing'], normalize='index') * 100
print("\nPorcentajes por tipo:")
print(tabla_porcentajes)

`duration` Las llamadas (call): Siempre tienen duración (17,908 casos con duración, 0 sin duración). Los mensajes (text): Casi nunca tienen duración (solo 16 casos con duración de 22,092 total).
Para `length` Las llamadas (call): Casi nunca tienen longitud (solo 12 casos de 17,908 total = 0.067%). Los mensajes (text): Siempre tienen longitud (22,092 casos con longitud, 0 sin longitud = 100%).
Esto confirma que los valores faltantes en ambos casis SÍ dependen de la variable `type`, lo cual es un patrón MAR perfectamente lógico desde el punto de vista del negocio. Los pocos mensajes de texto que sí tienen duración (16 casos) podrían ser errores en los datos o casos especiales como mensajes de voz.Se mantendrán estos valores como nulos en los dos casos.

---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico.

In [ ]:
# Columnas auxiliares
usage["is_text"] = (usage["type"] == "text").astype(int) #conocer el total de mensajes
usage["is_call"] = (usage["type"] == "call").astype(int) #conocer el total de llamadas


# Agrupar información por usuario
usage_agg = usage.groupby('user_id').agg({
    'is_text': 'sum',
    'is_call':'sum',
    'duration':'sum'
}).reset_index()

# observar resultado
usage_agg.head(3)

In [ ]:
# Renombrar columnas
usage_agg.rename(columns={'is_text':'cant_mensajes','is_call':'cant_llamadas','duration':'cant_minutos_llamada'}, inplace=True)
# observar resultado
usage_agg.head(3)

In [ ]:
# Combinar la tabla agregada con el dataset de usuarios
user_profile = users.merge(usage_agg,on=['user_id'],how='left')
user_profile.head(5)

### 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

In [ ]:
# Resumen estadístico de las columnas numéricas
user_profile.describe()

In [ ]:
# Distribución porcentual del tipo de plan
user_profile['plan'].value_counts(normalize=True)*100

---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

In [ ]:
# Histograma para visualizar la edad (age)
sns.histplot(data=user_profile, x='age', bins=10, palette=['skyblue', 'green'], hue='plan')
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.title('Distribución de Edad por Plan')
plt.show()

💡Insights:
- El Plan Básico (azul) representa la mayoría de usuarios en todos los grupos de edad
Esto sugiere que la mayoría de clientes prefieren opciones más económicas.

Se observan algunos patrones por edad:
- Usuarios jóvenes (20-40 años): Mayor proporción de Plan Premium (verde) comparado con otros grupos, los jóvenes están más dispuestos a pagar por servicios premium.
- Usuarios mayores (60+ años): Incremento notable en Plan Premium en el grupo 70-80 años, un posible segmento de alto valor adquisitivo.

Una oportunidad comercial clave: El grupo 40-50 años muestra el pico más alto de usuarios, pero con baja penetración de Premium.

In [ ]:
# Histograma para visualizar la cant_mensajes
sns.histplot(data=user_profile, x='cant_mensajes', bins=10, palette=['skyblue', 'green'], hue='plan')
plt.xlabel('Cantidad de mensajes')
plt.ylabel('Frecuencia')
plt.title('Distribución de la Cantidad de Mensajes por Plan')
plt.show()

💡Insights:
- Distribución sesgada a la derecha (Plan Premium)
- En el Plan Básico (azul): Pocos usuarios superan los 7.5 mensajes. En Plan Premium (verde): Distribución más amplia de uso (0-10 mensajes) y usuarios más activos en el rango 2.5-7.5 mensajes.

Dentro del plan Premium, hay mayor proporción de cantidad de mensajes que en la porción de plan básico.

In [ ]:
# Histograma para visualizar la cant_llamadas
sns.histplot(data=user_profile, x='cant_llamadas', bins=10, palette=['skyblue', 'green'], hue='plan')
plt.xlabel('Cantidad de llamadas')
plt.ylabel('Frecuencia')
plt.title('Distribución de Llamadas por Plan')
plt.show()

💡Insights:
- En general, los usuarios Premium hacen más llamadas.
- El plan Básico concentra más usuarios en rangos más bajos (2–5 llamadas).
- Los usuarios Básico tienden a hacer menos llamadas y enviar menos mensajes.

In [ ]:
# Histograma para visualizar la cant_minutos_llamada
sns.histplot(data=user_profile, x='cant_minutos_llamada', bins=10, palette=['skyblue', 'green'], hue='plan')
plt.xlabel('Cantidad de Minutos por Llamada')
plt.ylabel('Frecuencia')
plt.title('Distribución Total de Minutos por Plan')
plt.show()

💡Insights:
- Distribución sesgada a la derecha.
- La mayoría de usuarios está en rangos bajos (0–30 minutos), pero hay pocos casos con consumos muy altos (hasta ~150 min).
- Premium no solo hace más llamadas, sino que también acumula más minutos totales.

### 5.2 Identificación de Outliers

🎯 **Objetivo:**  
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

In [ ]:
# Visualizando usando BoxPlot
columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']

for col in columnas_numericas:
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=user_profile, y=col, palette=['skyblue'])
    plt.title(f'Boxplot: {col}')
    plt.ylabel(col)
    plt.show()

💡Insights:
- Age: no presenta outliers
- cant_mensajes: presenta outliers
- cant_llamadas: presenta outliers
- cant_minutos_llamada: presenta outliers

In [ ]:
# Calcular límites con el método IQR
columnas_limites = ['cant_mensajes','cant_llamadas','cant_minutos_llamada']

# Bucle para calcular límites IQR
for columna in columnas_limites:
    # Calcular Q1 y Q3
    Q1 = user_profile[columna].quantile(0.25)
    Q3 = user_profile[columna].quantile(0.75)

    # Calcular IQR
    IQR = Q3 - Q1

    # Calcular límites
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR

    # Mostrar resultados
    print(f"Columna: {columna}")
    print(f"Q1: {Q1:.2f}")
    print(f"Q3: {Q3:.2f}")
    print(f"IQR: {IQR:.2f}")
    print(f"Límite inferior: {limite_inferior:.2f}")
    print(f"Límite superior: {limite_superior:.2f}")
    print("-" * 40)



In [ ]:
# Revisa los limites superiores y el max, para tomar la decisión de mantener los outliers o no
user_profile[columnas_limites].describe()

💡Insights:
- cant_mensajes: mantener outliers.
- cant_llamadas: mantener outliers.
- cant_minutos_llamada: mantener outliers.

Porque representan segmentos valiosos, los valores no son imposibles y pueden ser importantes para estrategias comerciales.

---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

In [ ]:
# Crear columna grupo_uso
user_profile['grupo_uso'] = np.where(
    (user_profile['cant_llamadas'] < 5) & (user_profile['cant_mensajes'] < 5), 'Bajo uso',
    np.where(
        (user_profile['cant_llamadas'] < 10) & (user_profile['cant_mensajes'] < 10), 'Uso medio',
        'Alto uso'
    )
)

In [ ]:
# verificar cambios
user_profile.head()

### 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.


In [ ]:
# Crear columna grupo_edad
user_profile['grupo_edad'] = np.where(
    (user_profile['age'] <30), 'Joven',
    np.where(
        (user_profile['age'] < 60), 'Adulto',
        'Adulto mayor'
    )
)

In [ ]:
# verificar cambios
user_profile.head()

### 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

In [ ]:
# Visualización de los segmentos por uso
sns.countplot(data=user_profile, y='grupo_uso')
plt.title('Segmentación de Clientes por Uso')
plt.xlabel('Cantidad de Usuarios')
plt.ylabel('Categoría de Uso')
plt.show()

In [ ]:
# Visualización de los segmentos por edad
sns.countplot(data=user_profile, y='grupo_edad')
plt.title('Segmentación de Clientes por Edad')
plt.xlabel('Cantidad de Usuarios')
plt.ylabel('Categoría de Edad')
plt.show()


---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

### Análisis ejecutivo

⚠️ **Problemas detectados en los datos**
1. Valores Sentinels (valores imposibles)
En la columna `age`:
- Problema: Valores de -999 (edad imposible)
- Cantidad: 55 valores (1.38% del total de 4,000 usuarios)
- Solución aplicada: Reemplazo con la mediana (47 años)
En la columna `city`:
- Problema: Valores "?"
- Cantidad: 96 filas (2.4% del total de 4,000 usuarios)
- Solución aplicada: Reemplazó con pd.NA

2. Fechas imposibles
En la columna `reg_date`:
- Problema: Fechas del año 2026 (futuro imposible)
- Cantidad: 40 filas (1% del total de 4,000 usuarios)
- Solución aplicada: Remplazó con pd.NaT

3. Valores nulos estructurales (MAR - Missing At Random)
En `duration` y `length`:
- Problema: Valores faltantes que dependían de la columna `type`
- Porcentajes:
  - `duration`: 55.19% de valores nulos (normal, porque los mensajes no tienen duración)
  - `length`: 44.74% de valores nulos (normal, porque las llamadas no tienen longitud)
- Solución aplicada: Se mantuvieron como nulos


🔍 **Segmentos por Edad**

Adultos (30-60 años) - Mayoría
- Comportamiento: Grupo más numeroso
- Insight: Baja penetración de Premium en el pico 40-50 años

Jóvenes (<30 años)
- Comportamiento: Mayor proporción de Plan Premium
- Insight: Más dispuestos a pagar por servicios premium

Adultos Mayores (60+ años)
- Comportamiento: Incremento notable de Premium en 70-80 años
- Insight: Segmento de alto valor adquisitivo


📊 **Segmentos por Nivel de Uso**

Uso Medio (~3,000 usuarios - 75%)
- Comportamiento: Entre 5-10 llamadas y 5-10 mensajes
- Características: Segmento dominante, uso equilibrado de servicios

Bajo Uso (~750 usuarios - 18.75%)
- Comportamiento: Menos de 5 llamadas y menos de 5 mensajes
- Características: Usuarios conservadores o esporádicos

Alto Uso (~250 usuarios - 6.25%)
- Comportamiento: Más de 10 llamadas o más de 10 mensajes
- Características: Usuarios intensivos, posibles candidatos Premium

Patrones Clave Identificados
- Premium: Más llamadas, más mensajes, más minutos totales
- Básico: Concentrado en rangos bajos de uso

➡️ El 75% de usuarios están en "Uso Medio" pero muchos tienen plan Básico esto sugiere: Gran potencial de upgrade a Premium. Además los Jóvenes (<30) ya adoptan Premium más fácilmente y Adultos mayores (70-80) muestran alta disposición de pago se podrían implementar estrategias diferenciadas por grupo etario

💡 **Recomendaciones**
1. Estrategia de Conversión Masiva en Usuarios de "Uso Medio" con Plan Básico (~2,250 usuarios) por ejemplo: "tu uso actual sugiere que Premium te ahorraría dinero"
2. En Usuarios de "Alto Uso" (250 usuarios) tener un programa VIP, soporte prioritario y funciones exclusivas.
3. Monitorear usuarios que reduzcan su uso >30% mes a mes como un sistema de alertas tempranas.